# 02 · Day-ahead load forecast
**Forecast definition:** predict each hour's ERCOT demand one day ahead. Every feature must be known a day earlier: calendar fields, demand lags of 24 h and 168 h, and temperature.
**Temperature:** load-weighted average of 8 Texas cities (weights = each ERCOT weather zone's share of 2019–2023 load).
**Limitation:** temperature is *observed* temperature standing in for a day-ahead weather forecast, so our error is optimistic. ERCOT's own forecast, which we compare against, had to use real weather forecasts.
**Split:** train 2019–2023, test 2024–2025 (by time, never random). The temperature definition is chosen on a 2023 validation year, never on the test set.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()

In [2]:
df = q("SELECT * FROM model_hourly ORDER BY period")
df = df.iloc[168:].copy()          # first 168 rows have no lag_168
df["cooling_degree"] = (df.avg_temp_c - 18.3).clip(lower=0)
df["heating_degree"] = (18.3 - df.avg_temp_c).clip(lower=0)
df["temp_sq"] = df.avg_temp_c ** 2
df["temp_delta_24"] = df.avg_temp_c - df.temp_lag_24
df["is_weekend"] = df.is_weekend.astype(int)
df["is_holiday"] = df.is_holiday.astype(int)
print(len(df), "rows;", df[["demand_mwh", "demand_lag_24", "demand_lag_168", "avg_temp_c"]].isna().sum().sum(), "nulls in key columns")
train = df[df.period.dt.year <= 2023].copy()
test = df[df.period.dt.year >= 2024].copy()
print(f"train {len(train):,} rows ({train.period.min():%Y-%m-%d} to {train.period.max():%Y-%m-%d})")
print(f"test  {len(test):,} rows ({test.period.min():%Y-%m-%d} to {test.period.max():%Y-%m-%d})")

61200 rows; 0 nulls in key columns
train 43,656 rows (2019-01-08 to 2023-12-31)
test  17,544 rows (2024-01-01 to 2025-12-31)


/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30464/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


## Features

In [3]:
TEMP = ["avg_temp_c", "cooling_degree", "heating_degree", "temp_sq"]
FLAGS = ["is_weekend", "is_holiday"]
LAGS = ["demand_lag_24", "demand_lag_168"]
CAL = ["local_hour", "day_of_week", "month"]
BASE_FEATURES = TEMP + FLAGS + LAGS + CAL
# The change-vs-yesterday model gets NO load-level inputs: yesterday's load is only the anchor that the
# predicted change is added to. With levels as inputs, the model learned 2019-23 mean reversion ("high load
# days are followed by drops") and, with load now permanently higher, under-forecast 2024-25 by ~1,000 MWh.
# Dropping them was chosen on the 2023 validation year (MAPE 2.92% -> 2.65%), not on the test set.
FINAL_FEATURES = TEMP + FLAGS + CAL + ["temp_delta_24"]

## Which temperature? (validation: train 2019–2022, validate 2023)

In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor
def temp_features(d, col):
    d = d.copy()
    d["avg_temp_c"] = d[col]
    d["cooling_degree"] = (d.avg_temp_c - 18.3).clip(lower=0)
    d["heating_degree"] = (18.3 - d.avg_temp_c).clip(lower=0)
    d["temp_sq"] = d.avg_temp_c ** 2
    d["temp_delta_24"] = d.avg_temp_c - d[col].shift(24)
    return d.dropna(subset=["temp_delta_24"])

rows = {}
for col, label in [("temp_4city_c", "4 cities, simple average"),
                   ("temp_8city_simple_c", "8 cities, simple average"),
                   ("avg_temp_c", "8 cities, load-weighted (used)")]:
    d = temp_features(df, col)
    tr, va = d[d.period.dt.year <= 2022], d[d.period.dt.year == 2023]
    m = HistGradientBoostingRegressor(random_state=0).fit(tr[FINAL_FEATURES], tr.demand_mwh - tr.demand_lag_24)
    e = va.demand_mwh - (va.demand_lag_24 + m.predict(va[FINAL_FEATURES]))
    rows[label] = {"RMSE (MWh)": round(np.sqrt((e ** 2).mean())), "MAPE (%)": round(100 * (e.abs() / va.demand_mwh).mean(), 2)}
pd.DataFrame(rows).T

,RMSE (MWh),MAPE (%)
"4 cities, simple average",1878.0,2.65
"8 cities, simple average",1914.0,2.74
"8 cities, load-weighted (used)",1871.0,2.65


## Models

In [5]:
import statsmodels.api as sm
from sklearn.ensemble import HistGradientBoostingRegressor

def metrics(y, p):
    e = y - p
    return {"RMSE (MWh)": np.sqrt((e ** 2).mean()), "MAPE (%)": 100 * (e.abs() / y).mean()}

# Compare every model on the same hours: drop the 48 test hours with no ERCOT forecast (Dec 5-7 2025)
n_before = len(test)
test = test[test.ercot_forecast_mwh.notna()].copy()
print(f"test rows used: {len(test):,} (dropped {n_before - len(test)} with no ERCOT forecast)")

preds = {}
# 1. Baseline: same hour one week earlier
preds["Baseline: same hour last week"] = test.demand_lag_168
# Extra reference: same hour yesterday
preds["Reference: same hour yesterday"] = test.demand_lag_24

# 2. Linear regression. Calendar fields as dummies. avg_temp_c is dropped because
#    cooling_degree - heating_degree = avg_temp_c - 18.3 (perfect collinearity), and
#    is_weekend is dropped because the day_of_week dummies already encode it.
OLS_NUM = ["cooling_degree", "heating_degree", "temp_sq", "temp_delta_24", "is_holiday"] + LAGS
def design(d, cols=None):
    X = pd.concat([d[OLS_NUM], pd.get_dummies(d[CAL].astype(str), drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant="add")
    return X if cols is None else X.reindex(columns=cols, fill_value=0)
train_ols = train.dropna(subset=["temp_delta_24"])
X_train = design(train_ols)
ols = sm.OLS(train_ols.demand_mwh, X_train).fit()
preds["Linear regression (OLS)"] = ols.predict(design(test, X_train.columns))
print(f"OLS R^2 on train: {ols.rsquared:.3f}")

# 3a. Gradient boosting, default settings, predicting load directly
gb_default = HistGradientBoostingRegressor(random_state=0).fit(train[BASE_FEATURES], train.demand_mwh)
preds["Gradient boosting (default)"] = gb_default.predict(test[BASE_FEATURES])

# 3b. Gradient boosting, default settings, predicting the change from the same hour yesterday.
#     Trees cannot extrapolate past the training range; anchoring on yesterday's load lets the
#     model handle 2024-25 load levels it never saw in training.
gb = HistGradientBoostingRegressor(random_state=0).fit(
    train_ols[FINAL_FEATURES], train_ols.demand_mwh - train_ols.demand_lag_24)
preds["Gradient boosting (change vs yesterday)"] = test.demand_lag_24 + gb.predict(test[FINAL_FEATURES])

# 4. The grid operator's own day-ahead forecast (ERCOT, as filed with EIA on Form EIA-930)
preds["ERCOT official day-ahead forecast"] = test.ercot_forecast_mwh

results = pd.DataFrame({k: metrics(test.demand_mwh, v) for k, v in preds.items()}).T.round(2)
results["RMSE (MWh)"] = results["RMSE (MWh)"].round(0).astype(int)
results

test rows used: 17,496 (dropped 48 with no ERCOT forecast)
OLS R^2 on train: 0.948


,RMSE (MWh),MAPE (%)
Baseline: same hour last week,6481,8.52
Reference: same hour yesterday,3500,4.51
Linear regression (OLS),3604,5.56
Gradient boosting (default),3733,5.70
Gradient boosting (change vs yesterday),1971,2.52
ERCOT official day-ahead forecast,1756,2.31


In [6]:
ours = [k for k in results.index if not k.startswith(("Baseline", "Reference", "ERCOT"))]
best_name = results.loc[ours, "MAPE (%)"].idxmin()
print("best model of ours:", best_name)
test["pred_best"] = preds[best_name]
test["pred_baseline"] = preds["Baseline: same hour last week"]
test["ape_best"] = 100 * (test.demand_mwh - test.pred_best).abs() / test.demand_mwh
test["ape_baseline"] = 100 * (test.demand_mwh - test.pred_baseline).abs() / test.demand_mwh
test["ape_ercot"] = 100 * (test.demand_mwh - test.ercot_forecast_mwh).abs() / test.demand_mwh
results.to_csv("data/clean/model_results.csv")
test[["period", "local_ts", "local_hour", "season", "is_holiday", "demand_mwh", "avg_temp_c",
      "pred_best", "pred_baseline", "ercot_forecast_mwh", "ape_best", "ape_baseline", "ape_ercot"]].to_parquet("data/clean/test_predictions.parquet", index=False)

best model of ours: Gradient boosting (change vs yesterday)


## Our model vs ERCOT, head to head

In [7]:
daily = test.groupby(test.local_ts.dt.date).agg(ours=("ape_best", "mean"), ercot=("ape_ercot", "mean"),
                                                min_temp=("avg_temp_c", "min"), max_temp=("avg_temp_c", "max"))
print(f"days where our model beats ERCOT: {100 * (daily.ours < daily.ercot).mean():.1f}% of {len(daily)}")
print(f"hours where our model beats ERCOT: {100 * (test.ape_best < test.ape_ercot).mean():.1f}%")
print(f"bias (forecast - actual): ERCOT {(test.ercot_forecast_mwh - test.demand_mwh).mean():+.0f} MWh, ours {(test.pred_best - test.demand_mwh).mean():+.0f} MWh")
display(test.groupby(test.period.dt.year)[["ape_best", "ape_ercot"]].mean().round(2))
gap = daily.assign(gap=daily.ours - daily.ercot)
print("Days ERCOT beat us by the most:")
display(gap.sort_values("gap", ascending=False).head(5).round(1))
print("Days we beat ERCOT by the most:")
gap.sort_values("gap").head(5).round(1)

days where our model beats ERCOT: 45.7% of 731
hours where our model beats ERCOT: 48.4%
bias (forecast - actual): ERCOT +417 MWh, ours +51 MWh


,ape_best,ape_ercot
period,,
2024,2.63,2.16
2025,2.41,2.46


Days ERCOT beat us by the most:


,ours,ercot,min_temp,max_temp,gap
local_ts,,,,,
2024-01-15,14.3,4.8,-7.1,0.8,9.5
2024-11-05,10.1,2.2,14.8,21.8,7.9
2025-01-02,8.8,1.8,6.2,16.3,7.0
2024-01-18,13.8,7.0,4.8,17.5,6.8
2024-11-01,9.2,2.7,18.6,26.2,6.5


Days we beat ERCOT by the most:


,ours,ercot,min_temp,max_temp,gap
local_ts,,,,,
2025-12-02,2.6,10.0,1.1,11.3,-7.4
2025-12-01,2.7,8.4,4.9,7.9,-5.7
2025-10-30,1.5,6.5,7.3,19.5,-5.1
2025-11-10,3.8,8.8,4.1,14.7,-5.0
2024-06-05,3.9,8.3,24.6,31.3,-4.4


## ERCOT API: the 9:30 am day-ahead posting, every forecast model
ERCOT re-posts its 7-day forecast every hour. `src/pull_ercot.py` keeps, for each delivery day, the posting made at 09:30 Central the day before (the last one before the 10:00 day-ahead market closes), for all of ERCOT's forecast models, with the model ERCOT flagged as in use.

In [8]:
display(q("SELECT * FROM ercot_model_accuracy"))
display(q("SELECT * FROM ercot_model_in_use"))
api = q("SELECT period, system_total_mwh AS ercot_api_mwh FROM fact_ercot_forecast_model WHERE in_use")
api = api.groupby("period", as_index=False).ercot_api_mwh.mean()   # guard against two in-use rows
cmp = test.merge(api, on="period", how="inner")
print(f"hours with an ERCOT API forecast: {len(cmp):,} of {len(test):,}")
head = pd.DataFrame({
    "ERCOT 9:30 am posting, in-use model (ERCOT API)": metrics(cmp.demand_mwh, cmp.ercot_api_mwh),
    "ERCOT day-ahead forecast (EIA-930 copy)": metrics(cmp.demand_mwh, cmp.ercot_forecast_mwh),
    f"Ours: {best_name}": metrics(cmp.demand_mwh, cmp.pred_best),
    "Baseline: same hour last week": metrics(cmp.demand_mwh, cmp.pred_baseline),
}).T.round(2)
head.to_csv("data/clean/ercot_api_comparison.csv")
head

,forecast,mape_pct,rmse_mwh,bias_mwh,hours,first_day
0,X,2.08,1609.0,600.0,12402,2024-08-02
1,M,2.25,1719.0,20.0,17537,2024-01-01
2,In use (ERCOT's daily pick),2.39,1814.0,415.0,17537,2024-01-01
3,E3,2.58,1982.0,-156.0,17537,2024-01-01
4,E,2.60,1988.0,196.0,17537,2024-01-01
5,A3,2.62,1975.0,461.0,17537,2024-01-01
6,A6,2.71,2038.0,469.0,17537,2024-01-01
7,E2,2.72,1990.0,-43.0,17537,2024-01-01
8,E1,2.77,2100.0,-148.0,17537,2024-01-01


,model,hours_in_use,pct_of_hours
0,E,4956,28.2
1,E3,3648,20.8
2,E2,3223,18.4
3,E1,3067,17.5
4,X,1787,10.2
5,M,839,4.8
6,A3,24,0.1


hours with an ERCOT API forecast: 17,489 of 17,496


/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30464/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


,RMSE (MWh),MAPE (%)
"ERCOT 9:30 am posting, in-use model (ERCOT API)",1810.30,2.38
ERCOT day-ahead forecast (EIA-930 copy),1756.62,2.31
Ours: Gradient boosting (change vs yesterday),1970.87,2.52
Baseline: same hour last week,6482.33,8.53


## Does our model add anything to ERCOT's? A blend, weight chosen on 2024, tested on 2025
If a weighted average of our forecast and ERCOT's beats ERCOT alone, our model carries information ERCOT's forecast misses. The weight is fit on 2024 only, then scored on 2025.

In [9]:
def mape(y, p):
    return float(100 * ((y - p).abs() / y).mean())
c24, c25 = cmp[cmp.period.dt.year == 2024], cmp[cmp.period.dt.year == 2025]
grid = np.round(np.arange(0, 1.01, 0.05), 2)
fit = pd.Series({w: mape(c24.demand_mwh, w * c24.pred_best + (1 - w) * c24.ercot_api_mwh) for w in grid})
w = fit.idxmin()
blend25 = w * c25.pred_best + (1 - w) * c25.ercot_api_mwh
blend = pd.DataFrame({
    "ERCOT alone (API, in-use)": {"MAPE 2025 (%)": mape(c25.demand_mwh, c25.ercot_api_mwh)},
    "Ours alone": {"MAPE 2025 (%)": mape(c25.demand_mwh, c25.pred_best)},
    f"Blend: {w:.2f} ours + {1 - w:.2f} ERCOT": {"MAPE 2025 (%)": mape(c25.demand_mwh, blend25)},
}).T.round(2)
blend["weight on ours (fit on 2024)"] = [np.nan, np.nan, w]
blend.to_csv("data/clean/blend_2025.csv")
blend

,MAPE 2025 (%),weight on ours (fit on 2024)
"ERCOT alone (API, in-use)",2.53,NaN
Ours alone,2.41,NaN
Blend: 0.40 ours + 0.60 ERCOT,2.06,0.4


In [10]:
q("SELECT * FROM ercot_zone_accuracy")

,zone,zone_name,mape_pct,avg_load_mwh,hours
0,FWES,far_west,3.63,7103.0,15374
1,COAS,coast,3.74,14204.0,15374
2,SCEN,south_central,3.88,8907.0,15374
3,SOUT,southern,3.94,4350.0,15374
4,NCEN,north_central,3.96,15295.0,15374
5,EAST,east,5.72,1849.0,15374
6,NRTH,north,6.39,1797.0,15374
7,WEST,west,9.59,1401.0,15374


## Where does it fail?

In [11]:
by_hour = test.groupby("local_hour")[["ape_best", "ape_baseline", "ape_ercot"]].mean().round(2)
by_season = test.groupby("season")[["ape_best", "ape_baseline", "ape_ercot"]].mean().round(2).loc[["Winter", "Spring", "Summer", "Fall"]]
display(by_season); by_hour.T

,ape_best,ape_baseline,ape_ercot
season,,,
Winter,3.21,12.12,2.75
Spring,2.64,7.49,2.28
Summer,2.05,6.70,2.19
Fall,2.20,7.87,2.04


local_hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
ape_best,2.22,2.19,2.18,2.19,2.22,2.25,2.22,2.46,2.35,2.31,2.37,2.42,2.59,2.77,2.94,3.06,3.16,3.09,3.05,2.75,2.58,2.51,2.42,2.25
ape_baseline,8.46,8.52,8.59,8.62,8.55,8.57,8.48,8.35,8.21,7.93,7.79,7.86,8.18,8.6,9.04,9.28,9.25,9.03,8.77,8.66,8.58,8.46,8.4,8.38
ape_ercot,2.21,1.93,1.95,2.03,2.01,1.93,1.92,2.0,2.09,2.46,3.01,2.79,2.62,2.58,2.61,2.66,2.64,2.54,2.41,2.27,2.22,2.21,2.18,2.16


In [12]:
print("holiday vs non-holiday MAPE (best model):")
display(test.groupby("is_holiday").ape_best.mean().round(2))
worst_days = (test.groupby(test.local_ts.dt.date)
                  .agg(mape=("ape_best", "mean"), peak_mwh=("demand_mwh", "max"),
                       min_temp=("avg_temp_c", "min"), max_temp=("avg_temp_c", "max"), holiday=("is_holiday", "max"))
                  .sort_values("mape", ascending=False).head(10).round(1))
worst_days

holiday vs non-holiday MAPE (best model):


is_holiday
0    2.48
1    3.93
Name: ape_best, dtype: float64

,mape,peak_mwh,min_temp,max_temp,holiday
local_ts,,,,,
2024-07-08,14.9,55545.0,24.6,28.9,0
2024-01-15,14.3,76339.0,-7.1,0.8,1
2024-01-18,13.8,60103.0,4.8,17.5,0
2024-11-05,10.1,51222.0,14.8,21.8,0
2024-01-19,9.3,62802.0,-0.4,7.0,0
2024-11-01,9.2,54605.0,18.6,26.2,0
2025-01-20,9.1,73937.0,-4.0,2.2,1
2025-01-02,8.8,51821.0,6.2,16.3,0
2025-05-26,8.5,62105.0,22.3,27.0,1


In [13]:
import plotly.graph_objects as go
from viz import add_caption, save, SERIES
fig = go.Figure()
for i, (col, name) in enumerate([("ape_baseline", "Baseline (last week)"), ("ape_best", best_name), ("ape_ercot", "ERCOT official forecast")]):
    fig.add_bar(x=by_season.index, y=by_season[col], name=name, marker_color=SERIES[i])
fig.update_layout(title="Forecast error by season, test set 2024–2025", barmode="group", bargap=0.35, bargroupgap=0.08,
                  yaxis_title="Mean absolute % error", xaxis_title="Season")
add_caption(fig, "Winter is the hardest season for every forecast: cold snaps (and holidays such as MLK Day during them) drive the biggest misses.")
save(fig, "model_error_by_season")